### StateGraph 参数详解
```python
StateGraph(
    self,
    state_schema,
    context_schema,
    *,
    input_schema,
    output_schema
    )
```
- - -
##### state_schema（必填）
定义整个图(Graph)在运行过程中共享的状态结构  
通常使用`TypedDict`, `dataclass`, 或Pydantic `BaseModel`定义

- - -

##### context_schema(可选)
定义运行时上下文(Runtime Context), 用于传递只读、不需要持久化的配置或依赖
- 不可变（immutable）：在一次 invoke/stream 过程中不会被修改。
- 不进入 checkpoint：不会被保存到持久化内存中（适合放敏感信息、数据库连接、大对象等）。
- 常用于传递：API Key、数据库连接、当前用户 ID、模型配置、环境变量等。

- - -

##### input_schema(可选)
定义图的输入结构（即调用 graph.invoke() 时传入的数据格式）
通常是 `state_schema` 的一个子集。

- - -

##### output_schema（可选）
定义图的最终输出结构（即 invoke / stream 返回的结果）
同样可以是 `state_schema` 的子集。
常用于只返回用户关心的部分字段，隐藏内部中间状态。

In [7]:
from typing import Annotated, TypedDict
from langgraph.graph import START, END, MessagesState, StateGraph
from langgraph.runtime import Runtime

下面的例子通过运行时上下文来判断用户是否是授权用户，最终根据输入返回不同的信息

In [8]:
# 输入Schema
class InputSchema(TypedDict):
    question : str

# 输出Schema    
class OutputSchema(TypedDict):
    answer : str
    user_info : str

# 全局Schema
class OverallSchema(MessagesState):
    # input schema
    question : str

    # output schema
    answer : str
    user_info : str

# context schema
class ConfigSchema(TypedDict):
    user_id : int

In [9]:
def node_id_check(state:InputSchema, runtime:Runtime[ConfigSchema])->OverallSchema:
    if runtime.context.get('user_id') == 1:
        print('授权用户，允许访问')
        return {
            'user_info':'1'
        }
    else:
        print('未授权用户，禁止访问')
        return {
            'user_info': '-1'
        }

def node_info(state:OverallSchema)->OverallSchema:
    if state['user_info'] == '1':
        print(f"question:{state['question']}")
        e = '这是一条测试回答'
        print(f'answer:{e}')
        return {
            'answer' : e
        }
    else:
        return{
            'answer' : '--REJECT--'
        }



In [10]:
builder = StateGraph(
    state_schema = OverallSchema, 
    context_schema = ConfigSchema,
    input_schema = InputSchema,
    output_schema = OutputSchema)
builder.add_node('n1', node_id_check)
builder.add_node('n2', node_info)

builder.add_edge(START, 'n1')
builder.add_edge('n1', 'n2')
builder.add_edge('n2', END)


In [29]:
graph = builder.compile()

input = {
    'question': '请求测试用户权限'
}
config = {
    'user_id': 2 # 尝试 1
}
res = graph.invoke(
    input,
    context = config
)
print(res)

未授权用户，禁止访问
{'answer': '--REJECT--', 'user_info': '-1'}
